In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np

In [14]:
headers={'User-Agent':'Mozilla/5.0 (Windows NT 6.3; Win 64 ; x64) Apple WeKit /537.36(KHTML , like Gecko) Chrome/80.0.3987.162 Safari/537.36'} 
webpage=requests.get('https://www.ambitionbox.com/list-of-companies?page=1',headers=headers).text
soup=BeautifulSoup(webpage,'lxml')

In [26]:
len(soup.find_all('h1'))

1

In [27]:
soup.find_all('h1')[0].text

'\n\t\t\t\t\tTop Companies in\n\t\t\t\t \n\t\t\t\t\tINDIA\n\t\t\t\t\t'

# **TO FIND OUT NAMES OF THE COMPANIES**

In [29]:
for i in soup.find_all('h2'):
    print(i.text.strip())

Companies in India
TCS
Accenture
Wipro
Cognizant
Capgemini
HDFC Bank
Infosys
HCLTech
ICICI Bank
Tech Mahindra
Genpact
TP
Axis Bank
Jio
Concentrix Corporation
Amazon
Reliance Retail
iEnergizer
LTM Limited
HDB Financial Services
Popular Collections by Industries
Popular Collections by Cities
Popular Collections by Roles


In [56]:
len(soup.find_all('h2'))

24

# **Finding out the ratings**

In [57]:
len(soup.find_all('div',class_='rating_text rating_text--md'))

20

In [58]:
for i in soup.find_all('div',class_='rating_text rating_text--md'):
    print(i.text.strip())

3.3
3.7
3.6
3.7
3.6
3.8
3.5
3.4
4.0
3.3
3.6
3.9
3.6
4.4
3.5
3.9
3.9
4.5
3.6
3.9


**Finding reviews**

In [69]:
reviews = soup.find_all('span', class_='companyCardWrapper__ActionCount')
for i in reviews:
    print(i.text.strip())

1.2L
10.4L
11.4k
5.2k
10.9k
100
75.7k
7.2L
9.5k
20.6k
6.9k
49
66.7k
4.9L
7k
171
4.9k
121
63.1k
6.1L
6.6k
940
5.6k
102
55.2k
5L
5.7k
2.1k
3.8k
44
54.3k
1.5L
3.2k
379
3.3k
97
50.2k
5.3L
8.6k
3.2k
4.9k
120
47.4k
4L
4.6k
280
3.9k
74
46.9k
1.6L
3k
18
3.7k
73
44.5k
2.9L
4.7k
623
3.5k
79
43.7k
2.4L
4k
496
3.6k
76
39.9k
1L
2.3k
1.7k
1.9k
39
34.4k
1L
2k
196
2.2k
125
34.2k
61.1k
4.5k
--
2.5k
93
33.1k
1.3L
2k
10
3.2k
69
32.6k
1.6L
6.1k
76
3.4k
96
27.9k
74.6k
2k
100
1.9k
151
27.7k
25.2k
1.7k
65
811
29
27.3k
2L
3.5k
482
865
34
26.4k
49.3k
1.1k
2
1.4k
80


In [70]:
len(reviews)

120

## **About Company and Location**

In [113]:
info = soup.find_all('span',class_='companyCardWrapper__interLinking')

In [116]:
domain_list = []
location_list = []

for i in info:
    raw_text = i.text.strip()
    
    if '|' in raw_text:
        # Split into domain and location part
        domain, location_part = raw_text.split('|', 1)
        
        # Remove the '+... other locations' part by splitting on '+'
        primary_location = location_part.split('+')[0]
        
        domain_list.append(domain.strip())
        location_list.append(primary_location.strip())
    else:
        # Fallback if a card doesn't have the '|' delimiter
        domain_list.append(raw_text)
        location_list.append(None)



In [120]:
reviews = soup.find_all('span', class_='companyCardWrapper__ActionCount')
ratings = soup.find_all('div',class_='rating_text rating_text--md')
company_name = soup.find_all('h2')

# **For scraping from mulitple pages**

In [123]:
all_companies = []

for page_num in range(1, 11):
    url = f'https://www.ambitionbox.com/list-of-companies?page={page_num}'
    webpage = requests.get(url, headers=headers).text
    soup = BeautifulSoup(webpage, 'lxml')
    
    cards = soup.find_all('div', class_='companyCardWrapper')
    
    for card in cards:
        name = card.find('h2').text.strip() if card.find('h2') else None
        rating = card.find('div', class_='rating_text rating_text--md').text.strip() if card.find('div', class_='rating_text rating_text--md') else None
        
        # --- Extract Reviews ---
        # AmbitionBox usually puts the review count in a span with class companyCardWrapper__ActionCount
        review_tag = card.find('span', class_='companyCardWrapper__ActionCount')
        reviews = review_tag.text.strip() if review_tag else None
        
        # Interlinking info (Domain | Location)
        info_span = card.find('span', class_='companyCardWrapper__interLinking')
        domain, primary_loc = None, None
        
        if info_span:
            raw_text = info_span.text.strip()
            if '|' in raw_text:
                d, l = raw_text.split('|', 1)
                domain = d.strip()
                primary_loc = l.split('+')[0].strip()
            else:
                domain = raw_text
                
        # Append all fields including 'Reviews'
        all_companies.append({
            'Company_Name': name,
            'Rating': rating,
            'Reviews': reviews,          # <--- Added Reviews column here
            'Domain': domain,
            'Primary_Location': primary_loc
        })

df = pd.DataFrame(all_companies)
df.to_csv('ambitionbox_companies.csv', index=False)